# SignBridge — Notebook 2: YOLO Person, Hand & Face Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshiaradhya/SignBridge/blob/main/notebooks/02_yolo_detection.ipynb)

This notebook:
1. Samples Drive videos into a persistent YOLO dataset.
2. Creates starter person, hand, and face labels with MediaPipe, then verifies the dataset before training.
3. Fine-tunes a lightweight **YOLOv8n** model on GPU.
4. Evaluates mAP metrics and exports an optimized ONNX model.

In [ ]:
# Step 1: Environment Setup
!git clone https://github.com/joshiaradhya/SignBridge.git /content/SignBridge || (cd /content/SignBridge && git pull)
%cd /content/SignBridge
!pip install -q -r ml/colab-requirements.txt
!pip install -q ultralytics

In [ ]:
# Step 2: Mount Drive and build the persistent YOLO dataset if needed
# Each sampled frame receives starter person, hand, and face labels from MediaPipe.
from google.colab import drive
from pathlib import Path
import yaml
from ml.signbridge.datasets import export_yolo_yaml, sample_yolo_frames

drive.mount('/content/drive')
data_dir = Path('/content/drive/MyDrive/SignBridge_Data')
yolo_root = data_dir / 'yolo_dataset'
yolo_yaml = yolo_root / 'data.yaml'

REBUILD_YOLO_DATASET = False  # Set True after adding videos or improving labels.
required_dirs = [
    yolo_root / 'images' / split
    for split in ('train', 'val')
] + [
    yolo_root / 'labels' / split
    for split in ('train', 'val')
]
missing = [str(path) for path in [yolo_yaml, *required_dirs] if not path.exists()]
if missing or REBUILD_YOLO_DATASET:
    video_paths = sorted(
        path for path in data_dir.rglob('*')
        if path.suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv', '.webm'} and yolo_root not in path.parents
    )[:100]
    if len(video_paths) < 2:
        raise FileNotFoundError(f'Add at least two sign-language videos under {data_dir} before training.')
    stats = sample_yolo_frames(video_paths, yolo_root, frames_per_video=3)
    export_yolo_yaml(yolo_yaml, yolo_root)
    print(f'Created starter YOLO dataset from {len(video_paths)} videos: {stats}')

config = yaml.safe_load(yolo_yaml.read_text()) or {}
names = config.get('names')
class_names = list(names.values()) if isinstance(names, dict) else names
if class_names != ['person', 'hand', 'face']:
    raise ValueError("data.yaml must define person, hand, face in class order 0, 1, 2")

splits = ('train', 'val')
image_counts = {split: len(list((yolo_root / 'images' / split).glob('*'))) for split in splits}
label_files = {split: list((yolo_root / 'labels' / split).glob('*.txt')) for split in splits}
label_counts = {split: len(paths) for split, paths in label_files.items()}
annotated_counts = {split: sum(bool(path.read_text().strip()) for path in paths) for split, paths in label_files.items()}
if not all(image_counts.values()) or not all(label_counts.values()) or not all(annotated_counts.values()):
    raise ValueError(
        f'Each split needs images and non-empty YOLO labels. Images: {image_counts}; '
        f'label files: {label_counts}; annotated labels: {annotated_counts}'
    )
print(f'Using persistent dataset: {yolo_root}')
print(f'Images: {image_counts}; annotated labels: {annotated_counts}')

In [ ]:
# Step 3: Train YOLOv8n detector on GPU
from ultralytics import YOLO

# Initialize lightweight nano model
model = YOLO('yolov8n.pt')

# Train on sampled frames
results = model.train(
    data=str(yolo_yaml),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project='runs/yolo_sign',
    name='person_hand_face'
)

In [ ]:
# Step 4: Validate and Export to ONNX
metrics = model.val()
print('mAP50-95:', metrics.box.map)

# Export to ONNX
onnx_file = model.export(format='onnx', dynamic=True, opset=17)
print('Exported ONNX detector to:', onnx_file)
!cp {onnx_file} /content/drive/MyDrive/SignBridge_Data/sign_yolo_v1.onnx 2>/dev/null || true